# Job Recommendation — Embedding Improved (Option 2)

Mục tiêu: dùng **Sentence Embedding retrieval** làm lõi, sau đó **rerank** theo taxonomy / skill overlap / seniority và sinh **explanation**.

Notebook này là **IMPROVED pipeline** (khác baseline).

Improved: lấy top-N bằng embedding trước, rồi rerank lại bằng:
- taxonomy (ngành: backend/frontend/data…)
- seniority (junior/mid/senior…)
- skill overlap (kỹ năng trùng để giải thích)

=> Lấy Top-N rộng trước → rồi lọc, chấm điểm lại → chọn Top-K cuối → tinh hơn

## 0) Import & Config

In [1]:
import os, re
import numpy as np
import pandas as pd
from pathlib import Path
from sklearn.metrics.pairwise import cosine_similarity

pd.set_option("display.max_colwidth", 120)

# ===== Paths =====
BASE_DIR = Path(".")
DATA_DIR = BASE_DIR / "datasets"
OUT_DIR  = BASE_DIR / "outputs" / "jobrec_emb"
OUT_DIR.mkdir(parents=True, exist_ok=True)

# --- Use absolute paths directly (recommended) ---
JOBS_PATH = Path("/Users/nguyenduykhanh/Documents/GraduationProject/Recommendation_Project/recommendation/datasets/jobs_merged_it.xlsx")
CVS_PATH  = Path("/Users/nguyenduykhanh/Documents/GraduationProject/Recommendation_Project/recommendation/datasets/UpdatedResumeDataSet.csv")

# (Optional) If you copied files into ./datasets, use these instead:
# JOBS_PATH = DATA_DIR / "jobs_merged_it.xlsx"
# CVS_PATH  = DATA_DIR / "UpdatedResumeDataSet.csv"

MODEL_NAME = "sentence-transformers/all-MiniLM-L6-v2"
TOPN_RETRIEVE = 200
TOPK_FINAL    = 10

CACHE_DIR = OUT_DIR / "cache"
CACHE_DIR.mkdir(parents=True, exist_ok=True)

JOB_EMB_PATH = CACHE_DIR / f"job_embeddings_{MODEL_NAME.split('/')[-1]}.npy"
CV_EMB_PATH  = CACHE_DIR / f"cv_embeddings_{MODEL_NAME.split('/')[-1]}.npy"

# ===== Load data =====
jobs = pd.read_excel(JOBS_PATH)
resumes = pd.read_csv(CVS_PATH)

print("Jobs shape:", jobs.shape)
print("Resumes shape:", resumes.shape)
print("Jobs columns:", list(jobs.columns))
print("Resumes columns:", list(resumes.columns))

# ===== Ensure id exists (to avoid KeyError later) =====
# jobs_merged_it.xlsx usually doesn't have 'id'
if "id" not in jobs.columns:
    jobs = jobs.reset_index(drop=True)
    jobs["id"] = jobs.index.astype(int)

# ===== Schema adapter: create cleaned_description if missing =====
# merged jobs has 'description' not 'cleaned_description'
if "cleaned_description" not in jobs.columns:
    if "description" in jobs.columns:
        jobs["cleaned_description"] = jobs["description"]
    elif "job_description" in jobs.columns:
        jobs["cleaned_description"] = jobs["job_description"]
    else:
        raise ValueError("jobs missing a description column to build cleaned_description")

# ===== Basic cleaning =====
def clean_text(text: str) -> str:
    if not isinstance(text, str):
        return ""
    text = text.lower()
    text = text.replace("\n", " ").replace("\r", " ")
    # keep + # . if you care about c++, c#, node.js
    text = re.sub(r"[^a-z0-9\s\+\#\.]", " ", text)
    text = re.sub(r"\s+", " ", text)
    return text.strip()

# Clean metadata for nicer output
jobs["title"] = jobs.get("title", "").fillna("").astype(str).str.strip()
jobs["company"] = jobs.get("company", "Unknown").fillna("Unknown").astype(str).str.strip()
jobs["location"] = jobs.get("location", "").fillna("").astype(str).str.strip()

# Build job_texts for embedding
job_texts = (jobs["title"] + " " + jobs["cleaned_description"].fillna("").astype(str)).apply(clean_text)

# Build cv_texts for embedding (UpdatedResumeDataSet.csv uses Resume)
if "Resume" not in resumes.columns:
    raise ValueError("resumes dataset missing required column: Resume")
cv_texts = resumes["Resume"].fillna("").astype(str).apply(clean_text)

print("job_texts:", job_texts.shape, "cv_texts:", cv_texts.shape)


Jobs shape: (89277, 5)
Resumes shape: (962, 2)
Jobs columns: ['title', 'description', 'company', 'location', 'source']
Resumes columns: ['Category', 'Resume']
job_texts: (89277,) cv_texts: (962,)


## 1) Load data

- đọc it_jobs.xlsx + UpdatedResumeDataSet.csv

- check cột cần thiết

In [3]:
# kiểm tra file có tồn tại không
assert JOBS_PATH.exists(), f"Missing: {JOBS_PATH.resolve()}"
assert CVS_PATH.exists(), f"Missing: {CVS_PATH.resolve()}"

jobs = pd.read_excel(JOBS_PATH)
resumes = pd.read_csv(CVS_PATH)

# ===== SCHEMA ADAPTER FOR MERGED IT =====
# merged file có 'description' thay vì 'cleaned_description'
if "cleaned_description" not in jobs.columns:
    if "description" in jobs.columns:
        jobs["cleaned_description"] = jobs["description"]
    elif "job_description" in jobs.columns:
        jobs["cleaned_description"] = jobs["job_description"]
    else:
        raise ValueError("Jobs dataset missing a description column")

# đảm bảo các cột meta tồn tại
for col in ["title", "company", "location"]:
    if col not in jobs.columns:
        raise ValueError(f"Jobs missing column: {col}")

# ===== BASIC SANITY CHECKS (SAU KHI ADAPT) =====
required_job_cols = {"title","cleaned_description","company","location"}
missing = [c for c in required_job_cols if c not in jobs.columns]
if missing:
    raise ValueError(f"Jobs missing columns: {missing}")

if "Resume" not in resumes.columns:
    raise ValueError("Resumes missing column: Resume")

print("Jobs:", jobs.shape, "| Resumes:", resumes.shape)
jobs.head(3)


Jobs: (89277, 6) | Resumes: (962, 2)


,title,description,company,location,source,cleaned_description
0,Support Technologist II 2024-02216,**description and functions** ----------------------------- **open until filled** **general description:** this posi...,State of Wyoming,"Rock Springs, WY",it,**description and functions** ----------------------------- **open until filled** **general description:** this posi...
1,Computer Technician,"***if you would like to apply for this position, please submit your information through our official jobs site at: h...",Medford Township Public Schools,"Medford, NJ",it,"***if you would like to apply for this position, please submit your information through our official jobs site at: h..."
2,IT Technician,i need a sped up recording converted to text. the original is a youtube video. job type: contract pay: up to $24.91 ...,Law Office of Ruben and Ruben LLC,NaN,it,i need a sped up recording converted to text. the original is a youtube video. job type: contract pay: up to $24.91 ...


## 2) Normalize text

In [4]:
def norm_text(x) -> str:
    x = "" if pd.isna(x) else str(x)
    x = x.lower()
    x = x.replace("\n"," ").replace("\r"," ")
    x = re.sub(r"[^a-z0-9\s\+\#\.\-]", " ", x)  # keep + # . -
    x = re.sub(r"\s+", " ", x).strip()
    return x

jobs["title"] = jobs["title"].fillna("").astype(str)
jobs["cleaned_description"] = jobs["cleaned_description"].fillna("").astype(str)
jobs["company"] = jobs["company"].fillna("Unknown").astype(str)
jobs["location"] = jobs["location"].fillna("").astype(str)

job_texts = (jobs["title"] + " " + jobs["cleaned_description"]).map(norm_text)

cv_texts = resumes["Resume"].fillna("").astype(str).map(norm_text)

job_texts.iloc[0], cv_texts.iloc[0][:200]


('support technologist ii 2024-02216 description and functions ----------------------------- open until filled general description this position is an entry-level position that with guidance provides technical support technical expertise customer satisfaction and timeliness to assigned state agencies. the computer technology support specialist ii works in a team environment to provide tier 2 direct hardware and software support for state-issued desktops laptops printers voip-enabled devices and wireless mobile devices. this position works with higher-level technicians to complete projects both internally and for state agencies. the position troubleshoots information systems and determines the best course of action while utilizing team resources required to return the system to optimum performance. the incumbent will assist in the troubleshooting of information systems network telephony and firewall equipment with the guidance and oversight of specialized teams and teammates lead techni

## 3) Embedding (cache)

In [5]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer(MODEL_NAME)

if JOB_EMB_PATH.exists():
    job_embeddings = np.load(JOB_EMB_PATH)
    print("Loaded job embeddings:", job_embeddings.shape)
else:
    job_embeddings = model.encode(
        job_texts.tolist(),
        batch_size=64,
        show_progress_bar=True,
        normalize_embeddings=True
    )
    np.save(JOB_EMB_PATH, job_embeddings)
    print("Saved job embeddings:", JOB_EMB_PATH.resolve(), "| shape:", job_embeddings.shape)

# (Optional) precompute norms not needed since normalize_embeddings=True


/Users/nguyenduykhanh/Documents/GraduationProject/Recommendation_Project/.venv/lib/python3.9/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(
/Users/nguyenduykhanh/Documents/GraduationProject/Recommendation_Project/.venv/lib/python3.9/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loaded job embeddings: (21961, 384)


## 4) Retrieval (Embedding Top-N)
- Bước lọc thô để lấy ra danh sách job TIỀM NĂNG NHẤT cho 1 CV nhưng chưa có tinh lọc

* Các bước để triển khai bao gồm: 

    - Nhận vào 1 CV (cv_index)

    - Dùng model embedding đã load (ở cell trước)

    - Biến CV text đã được làm sạch → vector số

    - So sánh vector CV với vector của TẤT CẢ job

    - Tính độ tương đồng ngữ nghĩa (semantic_score) bằng cosine similarity

    - Sắp xếp job theo điểm từ cao → thấp

    - Lấy Top-N job có điểm cao nhất (topn, ví dụ 50 / 100)

    - Trả về bảng job TIỀM NĂNG gồm:

        - job_id

        - title

        - company

        - location

        - semantic_score

In [6]:
def retrieve_topn_jobs_for_cv(cv_index: int, topn: int = TOPN_RETRIEVE):
    cv_emb = model.encode([cv_texts.iloc[cv_index]], normalize_embeddings=True)
    scores = cosine_similarity(cv_emb, job_embeddings).flatten() 
    top_idx = np.argsort(scores)[::-1][:topn]

    cands = jobs.loc[top_idx].copy()
    cands["semantic_score"] = scores[top_idx] # seman chính là kết quả của độ tương đồng giữa cV và JD
    cands["job_id"] = cands.index.astype(str)
    return cands.reset_index(drop=True)

# quick sanity
retrieve_topn_jobs_for_cv(0, topn=5)[["job_id","title","company","location","semantic_score"]]


,job_id,title,company,location,semantic_score
0,21455,Salesforce Developer with CPQ Expertise,Concentrix,"Remote, US",0.556856
1,8317,Payor Network Coordinator - Onsite/hybrid role,"Edwards Health Care Services, Inc.","Hudson, OH, US",0.533593
2,21277,SCRUM Master,Data Innovations,"Colchester, VT, US",0.522664
3,10579,Kronos Consultant,ASM Tech Solutions,"Remote, US",0.520761
4,21421,Sr Applications Program Manager,Palomar Health,US,0.520613


## 5) Taxonomy mapping (CV Category → IT taxonomy)

> Đây là mapping **tạm**. Mục tiêu là tạo label để rerank và làm 

proxy-evaluation và có thể chỉnh dần theo taxonomy IT.

In [7]:
# If dataset has Category, use it. If not, fallback to 'unknown'
if "Category" not in resumes.columns:
    resumes["Category"] = "Unknown"

# ---- CV Category -> taxonomy (tune as you like) ----
CATEGORY_TO_TAX = { # ánh xạ CV cho ngành IT, chuẩn hoá nhãn
    "Data Science": "data",
    "Database": "data",
    "Business Analyst": "data",
    "Python Developer": "backend",
    "Java Developer": "backend",
    "DevOps Engineer": "devops",
    "Web Designing": "frontend",
    "Network Security Engineer": "security",
    "Testing": "qa",
    "HR": "non_it",
}

# danh mục ngành, nhãn ngành
def cv_taxonomy(cv_index: int) -> str:
    cat = str(resumes.loc[cv_index, "Category"])
    return CATEGORY_TO_TAX.get(cat, "other")

# ---- Job taxonomy classifier (title + desc keywords) ----
def classify_taxonomy_title_desc(title: str, desc: str) -> str:
    t = norm_text(title) + " " + norm_text(desc)

    # very rough rules
    if any(k in t for k in ["react", "vue", "angular", "frontend", "ui", "ux"]):
        return "frontend"
    if any(k in t for k in ["django", "flask", "fastapi", "spring", "node", "express", "backend", "api"]):
        return "backend"
    if any(k in t for k in ["data engineer", "warehouse", "etl", "airflow", "spark", "hadoop", "bi", "tableau", "power bi"]):
        return "data"
    if any(k in t for k in ["machine learning", "deep learning", "nlp", "computer vision", "pytorch", "tensorflow"]):
        return "ml"
    if any(k in t for k in ["devops", "kubernetes", "docker", "ci/cd", "jenkins", "terraform", "aws", "gcp", "azure"]):
        return "devops"
    if any(k in t for k in ["security", "soc", "penetration", "pentest", "siem"]):
        return "security"
    if any(k in t for k in ["qa", "tester", "testing", "automation test"]):
        return "qa"
    return "other"

# sanity check
print("CV0 tax:", cv_taxonomy(0))


CV0 tax: data


## 6) Seniority inference

- đoán level CV và Job để tránh gợi ý lệch kinh nghiệm - trình độ

- Mỗi CV có 1 level

- Mỗi Job có 1 level

- Bước sau sẽ:

    - so cv_sen vs job_sen

    - nếu lệch → trừ điểm

    - nếu khớp → giữ hoặc cộng điểm

In [8]:
def infer_seniority_from_title(title: str) -> str:
    t = norm_text(title)
    if any(x in t for x in ["intern", "internship"]): return "intern"
    if any(x in t for x in ["junior", "jr", "entry"]): return "junior"
    if any(x in t for x in ["director", "head", "principal", "lead", "senior", "sr"]): return "senior"
    return "mid"

def infer_seniority_from_cv(text: str) -> str:
    t = norm_text(text)
    if "intern" in t: 
        return "intern"
    m = re.search(r"\b(\d+)\+?\s*years\b", t)
    if m:
        y = int(m.group(1))
        if y >= 5: return "senior"
        if y >= 2: return "mid"
        return "junior"
    return "mid"

# quick
infer_seniority_from_title("Senior Backend Engineer"), infer_seniority_from_cv(cv_texts.iloc[0])


('senior', 'mid')

## 7) Skill extraction (explainability)

In [9]:
SKILLS = [
  # programming
  "python","java","c","c++","c#",".net","golang","go","php","ruby",
  "javascript","typescript",
  # data
  "sql","postgresql","mysql","mongodb","redis",
  "pandas","numpy","scipy","scikit-learn","sklearn","spark","hadoop","hive","airflow","etl","bi","tableau","power bi",
  # ml/ai
  "machine learning","deep learning","nlp","opencv","pytorch","tensorflow","keras",
  # web backend
  "django","flask","fastapi","spring","node","express","rest",
  # frontend
  "react","vue","angular","nextjs","next.js","html","css",
  # devops/cloud
  "docker","kubernetes","jenkins","gitlab","ci/cd","terraform","aws","gcp","azure",
]

# regex boundaries: keep skills like c++, c#, .net
def skill_pattern(skill: str) -> re.Pattern:
    s = re.escape(skill)
    # allow dot and plus and hash inside skill; use word-ish boundaries
    return re.compile(rf"(?<![a-z0-9]){s}(?![a-z0-9])")

SKILL_PATS = [(sk, skill_pattern(sk)) for sk in SKILLS]

def extract_skills(text: str) -> set:
    t = norm_text(text)
    found = set()
    for sk, pat in SKILL_PATS:
        if pat.search(t):
            found.add(sk)
    return found

# sanity
extract_skills("Worked with Python, SQL, AWS, Docker and React.")


{'aws', 'docker', 'python', 'react', 'sql'}

## 8) Rerank (Option 2) + Explanation
- Dùng để xếp hạng lại danh sách job đã retrieve

- Không tìm job mới, chỉ chấm điểm lại cho chính xác hơn

- Mục tiêu

    - Loại bớt job:

        - sai ngành

        - lệch trình độ

        - không có kỹ năng phù hợp

- Cách làm

    - Giữ semantic_score

    - Cộng thêm:

        - điểm kỹ năng

        - điểm ngành

    - Trừ điểm nếu lệch level

- Kết quả

    - Job cuối cùng:

        - đúng nghĩa

        - đúng ngành

        - đúng level

- Retrieve = lọc thô

- Rerank = tinh lọc

In [10]:
W_SEM   = 0.60  # trọng số semantic (độ tương đồng của job và cv)
W_SKILL = 0.25  # trọng số cho độ trùng kỹ năng
W_TAX   = 0.15  # trong số cho đúng ngành
W_SEN_P = 0.10  # trọng số trừ điểm nếu job cao level hơn CV

ORDER = {"intern":0, "junior":1, "mid":2, "senior":3}

# tính mức độ lệch do level trình độ để tránh gợi ý job quá cao so với level của CV
def seniority_penalty(cv_sen: str, job_sen: str) -> float:
    # penalty if job too senior compared to CV
    gap = ORDER.get(job_sen,2) - ORDER.get(cv_sen,1)
    if gap >= 2:
        return 1.0
    if gap == 1:
        return 0.3
    return 0.0


# lấy CV đang xét và danh sách job đã được lọc thô
def rerank_candidates(cv_index: int, cands: pd.DataFrame) -> pd.DataFrame:
    cv_text = cv_texts.iloc[cv_index]           # nội dung CV
    cv_tax  = cv_taxonomy(cv_index)             # ngành CV (backend, data,..)
    cv_sen  = infer_seniority_from_cv(cv_text)  # level CV
    cv_sk   = extract_skills(cv_text)           # skill CV
#   => hồ sơ UV

    out = cands.copy() # copy khong làm bẩn data gốc
    out["job_tax"] = out.apply(lambda r: classify_taxonomy_title_desc(r["title"], r["cleaned_description"]), axis=1)
    out["job_sen"] = out["title"].apply(infer_seniority_from_title)

    job_text_series = (out["title"] + " " + out["cleaned_description"]).map(norm_text)
    out["job_skills"] = job_text_series.apply(extract_skills)

    # tính điểm trùng kỹ năng
    def overlap(js):
        if not cv_sk:
            return 0.0
        return len(cv_sk & js) / max(1, len(cv_sk)) # (cv_sk & js) là kỹ năng trùng nhau
    out["skill_score"] = out["job_skills"].apply(overlap) # => job cùng strùng skill, điểm càng cao

    # tính điểm ngành
    out["tax_score"] = (out["job_tax"] == cv_tax).astype(float)

    # tính phạt lệch level: lệch càng nhiều - [hạt càng nặng]
    out["sen_penalty"] = out["job_sen"].apply(lambda js: seniority_penalty(cv_sen, js))

    out["final_score"] = (
        W_SEM * out["semantic_score"] +
        W_SKILL * out["skill_score"] +
        W_TAX * out["tax_score"] -
        W_SEN_P * out["sen_penalty"]
    )

    # explanation
    def explain(row):
        common = sorted(list(cv_sk & row["job_skills"]))
        missing = sorted(list(row["job_skills"] - cv_sk))  # note: from job side
        return common, missing

    ex = out.apply(explain, axis=1, result_type="expand")
    out["matched_skills"] = ex[0]   # skill cv và job cùng có
    out["job_extra_skills"] = ex[1] # skill job cần thêm

    return out.sort_values("final_score", ascending=False).reset_index(drop=True)

def recommend_improved(cv_index: int, topk: int = TOPK_FINAL) -> pd.DataFrame:
    cands = retrieve_topn_jobs_for_cv(cv_index, topn=TOPN_RETRIEVE)
    ranked = rerank_candidates(cv_index, cands).head(topk).copy()
    show_cols = ["job_id","title","company","location","semantic_score","final_score","job_tax","job_sen","matched_skills"]
    return ranked[show_cols]

recommend_improved(0, topk=10)


,job_id,title,company,location,semantic_score,final_score,job_tax,job_sen,matched_skills
0,12980,Software Platform and Product Engineer,Vimaan Robotics,"San Jose, CA, US",0.467428,0.444346,data,mid,[machine learning]
1,16846,RN - Neonatal ICU (NICU),HWS,"Denver, CO, USA",0.481392,0.438835,data,mid,[]
2,6571,Algorithmic Trading Software Analyst,Sugar Mill Farms,"Remote, US",0.455977,0.437475,data,mid,[machine learning]
3,5832,Salesforce Administrator,Stratos Wealth Partners,"Beachwood, OH, US",0.470498,0.432299,data,mid,[]
4,17352,IT Assistant,"Hot Springs Radiology Services, LTD","Hot Springs, AR, US",0.459261,0.425557,data,mid,[]
5,3940,Programmer Analyst,Virginia Tech,"Blacksburg, VA, US",0.499148,0.410600,frontend,mid,"[angular, css, docker, html, java, javascript, mysql, sql]"
6,8756,Sr. Salesforce Developer - Financial Services Cloud,KK Technologies,"Remote, US",0.478037,0.406822,data,senior,[]
7,20736,Systems Analyst,Dutech Systems Inc,"Austin, TX, US",0.504490,0.399916,frontend,mid,"[css, html, java, javascript, python, sql, tableau]"
8,2624,Systems Engineer - Manager - Star 2282-01,Integrated Intel Solutions,"McLean, VA, US",0.495600,0.366804,frontend,mid,"[angular, docker, java, mysql, python]"
9,4762,Test iCIMs Requisition,S&P Global,"Cambridge, MA, US",0.496499,0.353455,frontend,mid,"[docker, flask, machine learning, python]"


## 9) Export improved results (sample CVs)

In [11]:
def export_improved_for_cvs(cv_indices, topk=10):
    paths = []
    for i in cv_indices:
        df = recommend_improved(i, topk=topk)
        p = OUT_DIR / f"embedding_improved_cv{i}_top{topk}.csv"
        df.to_csv(p, index=False)
        paths.append(p)
    return paths

sample_cvs = [0, 5, 20, 100]
paths = export_improved_for_cvs(sample_cvs, topk=10)
paths


[PosixPath('outputs/jobrec_emb/embedding_improved_cv0_top10.csv'),
 PosixPath('outputs/jobrec_emb/embedding_improved_cv5_top10.csv'),
 PosixPath('outputs/jobrec_emb/embedding_improved_cv20_top10.csv'),
 PosixPath('outputs/jobrec_emb/embedding_improved_cv100_top10.csv')]

## 10) Proxy evaluation metrics (Option 2)

Do dataset không có ground-truth job↔cv, ta dùng **proxy metrics** để so sánh baseline vs improved:
- **TaxonomyHit@K**: tỉ lệ job cùng taxonomy với CV
- **SeniorityMismatch@K**: tỉ lệ job quá senior so với CV
- **AvgSkillOverlap@K**: trung bình % skill overlap (so với skill của CV)
- **ExplainabilityRate@K**: tỉ lệ job có ít nhất 1 skill trùng để giải thích

In [11]:
def baseline_topk(cv_index: int, top_k: int = TOPK_FINAL) -> pd.DataFrame:
    cands = retrieve_topn_jobs_for_cv(cv_index, topn=TOPN_RETRIEVE)
    b = cands.sort_values("semantic_score", ascending=False).head(top_k).copy()

    b["job_tax"] = b.apply(lambda r: classify_taxonomy_title_desc(r["title"], r["cleaned_description"]), axis=1)
    b["job_sen"] = b["title"].apply(infer_seniority_from_title)
    job_text_series = (b["title"] + " " + b["cleaned_description"]).map(norm_text)
    b["job_skills"] = job_text_series.apply(extract_skills)
    return b

def option2_metrics_for_cv(cv_index: int, df: pd.DataFrame) -> dict:
    cv_tax = cv_taxonomy(cv_index)
    cv_sen = infer_seniority_from_cv(cv_texts.iloc[cv_index])
    cv_sk  = extract_skills(cv_texts.iloc[cv_index])

    tax_hit = float((df["job_tax"] == cv_tax).mean()) if len(df) else 0.0

    def too_senior(js):
        return (ORDER.get(js,2) - ORDER.get(cv_sen,1)) >= 2
    sen_mismatch = float(df["job_sen"].apply(too_senior).mean()) if len(df) else 0.0

    def overlap(job_sk):
        if not cv_sk:
            return 0.0
        return len(cv_sk & job_sk) / max(1, len(cv_sk))
    avg_overlap = float(df["job_skills"].apply(overlap).mean()) if len(df) else 0.0

    expl_rate = float(df["job_skills"].apply(lambda js: len(cv_sk & js) > 0).mean()) if len(df) else 0.0

    return {
        "TaxonomyHit@K": tax_hit,
        "SeniorityMismatch@K": sen_mismatch,
        "AvgSkillOverlap@K": avg_overlap,
        "ExplainabilityRate@K": expl_rate,
    }

def evaluate_option2(sample_size: int = 100, top_k: int = 10, seed: int = 42):
    rng = np.random.default_rng(seed)
    idxs = rng.choice(len(resumes), size=min(sample_size, len(resumes)), replace=False)

    rows = []
    for i in idxs:
        b = baseline_topk(int(i), top_k=top_k)
        imp_full = rerank_candidates(int(i), retrieve_topn_jobs_for_cv(int(i), topn=TOPN_RETRIEVE)).head(top_k)
        mb = option2_metrics_for_cv(int(i), b)
        mi = option2_metrics_for_cv(int(i), imp_full)

        rows.append({"type":"baseline", "cv_index": int(i), **mb})
        rows.append({"type":"improved", "cv_index": int(i), **mi})

    df = pd.DataFrame(rows)
    summary = df.groupby("type")[["TaxonomyHit@K","SeniorityMismatch@K","AvgSkillOverlap@K","ExplainabilityRate@K"]].mean()
    return df, summary

metrics_df, summary_df = evaluate_option2(sample_size=100, top_k=10)
display(summary_df)

# save
metrics_df.to_csv(OUT_DIR / "embedding_option2_metrics_rows.csv", index=False)
summary_df.to_csv(OUT_DIR / "embedding_option2_metrics_summary.csv")
print("Saved metrics to:", OUT_DIR.resolve())


,TaxonomyHit@K,SeniorityMismatch@K,AvgSkillOverlap@K,ExplainabilityRate@K
type,,,,
baseline,0.073,0.314,0.169218,0.405
improved,0.134,0.230,0.458051,0.654


Saved metrics to: /Users/nguyenduykhanh/Documents/GraduationProject/Recommendation_Project/recommendation/notebooks/candidate/EBD/outputs/jobrec_emb
